# Individual Cases Analysis: Code

**Inputs:** CSV result file for individual cases (see `code/conflicting_changes/`)

**Inputs:** Provide the path to the results directory; the script assumes the results are in one CSV file, in a folder named `BASE/results/`. Update the data loading section to match your CSV file name.

In [ ]:
import re
import string
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings

warnings.filterwarnings("ignore")

BASE = "/XXX" # Add your base folder

# Set visual style
sns.set(style="whitegrid")
language_palette = {
    "english": "#73b6a1",
    "finnish": "#95a3c3",
    "swedish": "#e99675",
}

## 1. Data loading

In [ ]:
# Load data
df = pd.read_csv(f"{BASE}/results/XXX.csv", sep = ";") # Update to match your file

df

,model,case,perturbation,language,metric,value1,value2,value3
0,bge-m3,C3,modify,english,cosine_similarity,0.939970,0.939514,0.942920
1,bge-m3,C3,modify,english,euclidean_distance,0.346498,0.347811,0.337875
2,bge-m3,C3,modify,english,l1_distance,8.689926,8.789808,8.574211
3,embedding_gemma,C5,delete,english,cosine_similarity,0.984944,0.948819,0.949135
4,embedding_gemma,C5,delete,english,euclidean_distance,0.173526,0.319942,0.318951
...,...,...,...,...,...,...,...,...
61,ADA,C3,modify,finnish,l1_distance,3.178761,2.962431,3.121607
62,ADA,C4,delete,english,l1_distance,3.270435,3.266771,3.990294
63,ADA,C5,delete,swedish,cosine_similarity,0.991075,0.971790,0.973187
64,ADA,C5,delete,swedish,euclidean_distance,0.133602,0.234127,0.231744


In [72]:
df["language"] = df["language"].astype(str).str.capitalize()
df["perturbation"] = df["perturbation"].astype(str).str.capitalize()

model_name_map = {
    "3_large": "OpenAI 3 Large",
    "ada": "OpenAI ADA",
    "bge-m3": "BGE-M3",
    "snowflake-arctic-embed-l-v2": "Snowflake Arctic V2",
    "nomic-embed-text-v2": "Nomic Embed V2",
    "embedding_gemma": "EmbeddingGemma",
    "multilingual_e5_large": "Multilingual E5 Large",
    "qwen3-8b": "Qwen3-8B",
}

df["model"] = (
    df["model"]
    .astype(str)
    .str.lower()
    .map(model_name_map)
    .fillna(df["model"])  # fallback if something unexpected appears
)
df

,model,case,perturbation,language,metric,value1,value2,value3
0,BGE-M3,C3,Modify,English,cosine_similarity,0.939970,0.939514,0.942920
1,BGE-M3,C3,Modify,English,euclidean_distance,0.346498,0.347811,0.337875
2,BGE-M3,C3,Modify,English,l1_distance,8.689926,8.789808,8.574211
3,EmbeddingGemma,C5,Delete,English,cosine_similarity,0.984944,0.948819,0.949135
4,EmbeddingGemma,C5,Delete,English,euclidean_distance,0.173526,0.319942,0.318951
...,...,...,...,...,...,...,...,...
61,OpenAI ADA,C3,Modify,Finnish,l1_distance,3.178761,2.962431,3.121607
62,OpenAI ADA,C4,Delete,English,l1_distance,3.270435,3.266771,3.990294
63,OpenAI ADA,C5,Delete,Swedish,cosine_similarity,0.991075,0.971790,0.973187
64,OpenAI ADA,C5,Delete,Swedish,euclidean_distance,0.133602,0.234127,0.231744


In [73]:
expected_cols = [
    "model",
    "case",
    "perturbation",
    "language",
    "metric",
    "value1",
    "value2",
    "value3",
]
missing = set(expected_cols) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Ordering
case_order = [f"C{i}" for i in range(1, 6)]
pert_order = ["Delete", "Modify", "Paraphrase"]
level_cols = ["value1", "value2", "value3"]

df["case"] = pd.Categorical(df["case"], categories=case_order, ordered=True)
df["perturbation"] = pd.Categorical(
    df["perturbation"], categories=pert_order, ordered=True
)

# Which metrics are similarity-like (higher = more similar) vs distance-like (higher = more different)
similarity_metrics = {"cosine_similarity"} 
df["metric_type"] = np.where(
    df["metric"].isin(similarity_metrics), "similarity", "distance"
)
df

,model,case,perturbation,language,metric,value1,value2,value3,metric_type
0,BGE-M3,C3,Modify,English,cosine_similarity,0.939970,0.939514,0.942920,similarity
1,BGE-M3,C3,Modify,English,euclidean_distance,0.346498,0.347811,0.337875,distance
2,BGE-M3,C3,Modify,English,l1_distance,8.689926,8.789808,8.574211,distance
3,EmbeddingGemma,C5,Delete,English,cosine_similarity,0.984944,0.948819,0.949135,similarity
4,EmbeddingGemma,C5,Delete,English,euclidean_distance,0.173526,0.319942,0.318951,distance
...,...,...,...,...,...,...,...,...,...
61,OpenAI ADA,C3,Modify,Finnish,l1_distance,3.178761,2.962431,3.121607,distance
62,OpenAI ADA,C4,Delete,English,l1_distance,3.270435,3.266771,3.990294,distance
63,OpenAI ADA,C5,Delete,Swedish,cosine_similarity,0.991075,0.971790,0.973187,similarity
64,OpenAI ADA,C5,Delete,Swedish,euclidean_distance,0.133602,0.234127,0.231744,distance


In [74]:
import numpy as np
import pandas as pd

# df columns expected:
# ["model","case","perturbation","language","metric","value1","value2","value3"]

similarity_metrics = {"cosine_similarity"}  # extend if needed

def monotonic_violation_values(v1, v2, v3, metric_type):
    v = np.array([v1, v2, v3], dtype=float)
    if np.any(pd.isna(v)):
        return np.nan
    if metric_type == "similarity":
        return not (v[0] >= v[1] >= v[2])
    else:
        return not (v[0] <= v[1] <= v[2])

# 1) Build one wide table per (model, case, language, perturbation, metric)
supp_all = df.copy()

# metric type per row
supp_all["metric_type"] = np.where(supp_all["metric"].isin(similarity_metrics), "similarity", "distance")

# rename value columns to levels
supp_all = supp_all.rename(columns={"value1":"Level 1", "value2":"Level 2", "value3":"Level 3"})

# 2) Monotonicity indicator
supp_all["monotonicity_violated"] = supp_all.apply(
    lambda r: monotonic_violation_values(r["Level 1"], r["Level 2"], r["Level 3"], r["metric_type"]),
    axis=1
)

# 3) Optional: “change score” and deltas (helpful for auditability)
# change score = higher means more change
for L in ["Level 1", "Level 2", "Level 3"]:
    supp_all[f"chg_{L}"] = np.where(
        supp_all["metric_type"] == "similarity",
        1.0 - supp_all[L].astype(float),
        supp_all[L].astype(float)
    )

supp_all["delta_1_to_3"] = supp_all["chg_Level 3"] - supp_all["chg_Level 1"]
supp_all["delta_1_to_2"] = supp_all["chg_Level 2"] - supp_all["chg_Level 1"]
supp_all["delta_2_to_3"] = supp_all["chg_Level 3"] - supp_all["chg_Level 2"]

# 4) Pick final columns (compact vs verbose)
compact_cols = [
    "model","case","language","perturbation","metric",
    "Level 1","Level 2","Level 3",
    "monotonicity_violated"
]

verbose_cols = compact_cols + ["delta_1_to_3","delta_1_to_2","delta_2_to_3"]

# Choose one:
supp_table = supp_all[verbose_cols].copy()   # or use compact_cols

# --- FINAL exported columns (clean) ---
# Capitalize first letter, keep rest lowercase

export_cols = [
    "model",
    "case",
    "language",
    "perturbation",
    "metric",
    "Level 1",
    "Level 2",
    "Level 3",
]

supp_table = (
    supp_all[export_cols]
    .sort_values(["metric", "model", "case", "language", "perturbation"])
    .reset_index(drop=True)
)

# Preview
supp_table.head(20)


,model,case,language,perturbation,metric,Level 1,Level 2,Level 3
0,BGE-M3,C3,English,Modify,cosine_similarity,0.939970,0.939514,0.942920
1,EmbeddingGemma,C5,English,Delete,cosine_similarity,0.984944,0.948819,0.949135
2,EmbeddingGemma,C5,Finnish,Delete,cosine_similarity,0.988598,0.972551,0.975144
3,EmbeddingGemma,C5,Swedish,Delete,cosine_similarity,0.989829,0.972595,0.973122
4,Multilingual E5 Large,C2,Swedish,Delete,cosine_similarity,0.968508,0.960636,0.966842
5,Multilingual E5 Large,C3,English,Delete,cosine_similarity,0.980382,0.981212,0.976722
6,Multilingual E5 Large,C3,English,Modify,cosine_similarity,0.988417,0.992379,0.980220
7,Multilingual E5 Large,C3,Finnish,Modify,cosine_similarity,0.984843,0.985736,0.986952
8,Multilingual E5 Large,C4,Swedish,Modify,cosine_similarity,0.993779,0.994169,0.986723
9,Multilingual E5 Large,C5,Finnish,Modify,cosine_similarity,0.986209,0.991844,0.990320


In [75]:
# Excel (preferred by most medical informatics journals)
supp_table.to_excel(
    f"{BASE}/results/tables/Supplementary_Table_AllMetrics_CaseLevel.xlsx", index=False,
)

# CSV (optional)
supp_table.to_csv(
    f"{BASE}/results/tables/Supplementary_Table_AllMetrics_CaseLevel.csv", index=False
)
